# Geographic analysis

Analyses how municipality population relates to company type, filing coverage, financial performance, and distress indicators. It reads the curated table produced by `Build_analytics.ipynb`, so run that notebook first.

This is descriptive analysis. Population is measured at municipality level and the results do not establish causation.

In [1]:
import json
import os
from datetime import datetime, timezone

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

DATA_DIR = "/home/jovyan/data"
ANALYTICS_FILE = os.path.join(DATA_DIR, "parquet", "analytics_company_financials.parquet")
OUTPUT_FILE = os.path.join(DATA_DIR, "geography_analysis.json")

spark = SparkSession.builder.appName("group13_geographic_analysis").getOrCreate()
curated = spark.read.parquet(ANALYTICS_FILE)

required_columns = {
    "organisasjonsnummer", "organisasjonsform_kode",
    "municipality_population", "population_year",
    "har_regnskap", "konkurs", "under_avvikling",
    "operating_margin_pct", "revenue", "operating_income",
}
missing_columns = required_columns.difference(curated.columns)
assert not missing_columns, "Missing analytics columns: %s" % sorted(missing_columns)

n_rows = curated.count()
print("Rows read:", n_rows)
print("Population year(s):", [r[0] for r in curated.select("population_year").distinct().collect()])

Rows read: 1171373


Population year(s): [2026, None]


## Population bands

The bands make comparisons easier to interpret than raw municipality population. Companies without a matched municipality population remain in a separate category.

In [2]:
analysed = curated.withColumn(
    "population_band",
    F.when(F.col("municipality_population").isNull(), "missing")
     .when(F.col("municipality_population") < 5000, "under_5k")
     .when(F.col("municipality_population") < 20000, "5k_to_20k")
     .when(F.col("municipality_population") < 100000, "20k_to_100k")
     .otherwise("100k_plus"),
)

band_order = F.create_map(
    F.lit("under_5k"), F.lit(1),
    F.lit("5k_to_20k"), F.lit(2),
    F.lit("20k_to_100k"), F.lit(3),
    F.lit("100k_plus"), F.lit(4),
    F.lit("missing"), F.lit(5),
)

band_counts = (
    analysed.groupBy("population_band")
    .count()
    .withColumn("sort_order", band_order[F.col("population_band")])
    .orderBy("sort_order")
)
band_counts.show(truncate=False)

+---------------+------+----------+
|population_band|count |sort_order|
+---------------+------+----------+
|under_5k       |100610|1         |
|5k_to_20k      |211108|2         |
|20k_to_100k    |387799|3         |
|100k_plus      |408807|4         |
|missing        |63049 |5         |
+---------------+------+----------+



## Company type and filing coverage

This compares legal form across population bands. `filing_rate` is calculated over all registered entities, so missing filings remain part of the denominator.

In [3]:
legal_form_by_band = (
    analysed.groupBy("population_band", "organisasjonsform_kode")
    .agg(
        F.count("organisasjonsnummer").alias("entities"),
        F.sum(F.col("har_regnskap").cast("int")).alias("filed"),
    )
    .withColumn("filing_rate_pct", 100 * F.col("filed") / F.col("entities"))
    .orderBy("population_band", F.desc("entities"))
)
legal_form_by_band.show(40, truncate=False)

+---------------+----------------------+--------+------+------------------+
|population_band|organisasjonsform_kode|entities|filed |filing_rate_pct   |
+---------------+----------------------+--------+------+------------------+
|100k_plus      |AS                    |174885  |163149|93.28930440003431 |
|100k_plus      |ENK                   |162325  |1259  |0.7756044971507777|
|100k_plus      |FLI                   |32289   |1403  |4.345133017436279 |
|100k_plus      |ESEK                  |15504   |6312  |40.71207430340557 |
|100k_plus      |BRL                   |4488    |4291  |95.61051693404634 |
|100k_plus      |DA                    |3577    |480   |13.419066256639642|
|100k_plus      |KBO                   |3080    |0     |0.0               |
|100k_plus      |NUF                   |2769    |1152  |41.603466955579634|
|100k_plus      |STI                   |2518    |2358  |93.64575059571088 |
|100k_plus      |ANS                   |2412    |457   |18.9469320066335  |
|100k_plus  

## Financial performance and distress

Financial averages use filed rows only. Distress is defined here as either `konkurs` or `under_avvikling`; this is a descriptive indicator, not a causal outcome.

In [4]:
analysed = analysed.withColumn(
    "distress",
    F.coalesce(F.col("konkurs"), F.lit(False))
    | F.coalesce(F.col("under_avvikling"), F.lit(False))
)

financial_by_band = (
    analysed.filter(F.col("har_regnskap"))
    .groupBy("population_band")
    .agg(
        F.count("organisasjonsnummer").alias("filed_entities"),
        F.avg("operating_margin_pct").alias("mean_margin_pct"),
        F.expr("percentile_approx(operating_margin_pct, 0.5)").alias("median_margin_pct"),
        F.avg(F.col("revenue")).alias("mean_revenue"),
    )
    .orderBy("population_band")
)

distress_by_band = (
    analysed.groupBy("population_band")
    .agg(
        F.count("organisasjonsnummer").alias("entities"),
        F.sum(F.col("distress").cast("int")).alias("distressed"),
    )
    .withColumn("distress_rate_pct", 100 * F.col("distressed") / F.col("entities"))
    .orderBy("population_band")
)

financial_by_band.show(truncate=False)
distress_by_band.show(truncate=False)

+---------------+--------------+-------------------+-----------------+--------------------+
|population_band|filed_entities|mean_margin_pct    |median_margin_pct|mean_revenue        |
+---------------+--------------+-------------------+-----------------+--------------------+
|100k_plus      |181939        |-27306.276629000557|6.904851132248224|1.3557708488322628E8|
|20k_to_100k    |157725        |-3728.7033262400387|6.519068789086779|2.2338702840233855E7|
|5k_to_20k      |74047         |-632.8906227555618 |6.201651128909804|1.5810583264969792E7|
|missing        |2462          |-11.8494809236215  |7.739605508222601|9.934958936539368E7 |
|under_5k       |29186         |-592.314501691538  |5.644913387575142|1.172294963880538E7 |
+---------------+--------------+-------------------+-----------------+--------------------+



+---------------+--------+----------+-------------------+
|population_band|entities|distressed|distress_rate_pct  |
+---------------+--------+----------+-------------------+
|100k_plus      |408807  |4056      |0.9921552223910061 |
|20k_to_100k    |387799  |3556      |0.9169698735685239 |
|5k_to_20k      |211108  |1611      |0.7631165090853971 |
|missing        |63049   |11        |0.01744674776760932|
|under_5k       |100610  |709       |0.7047013219361893 |
+---------------+--------+----------+-------------------+



## Industry composition

This identifies the most common NACE level-1 industry codes in each population band. It is useful for spotting whether apparent geographic differences are actually industry-mix differences.

In [5]:
industry_window = Window.partitionBy("population_band").orderBy(F.desc("count"))

industry_by_band = (
    analysed.filter(F.col("naeringskode1_kode").isNotNull())
    .groupBy("population_band", "naeringskode1_kode", "naeringskode1_beskrivelse")
    .count()
    .withColumn("rank", F.row_number().over(industry_window))
    .filter(F.col("rank") <= 10)
    .orderBy("population_band", "rank")
)
industry_by_band.show(100, truncate=False)

+---------------+------------------+-------------------------------------------------------------------------------------------+-----+----+
|population_band|naeringskode1_kode|naeringskode1_beskrivelse                                                                  |count|rank|
+---------------+------------------+-------------------------------------------------------------------------------------------+-----+----+
|100k_plus      |00.000            |Uoppgitt                                                                                   |29183|1   |
|100k_plus      |68.200            |Utleie av egen eller leid fast eiendom                                                     |26987|2   |
|100k_plus      |97.001            |Aktiviteter i borettslag og boligsameier                                                   |21312|3   |
|100k_plus      |94.992            |Aktiviteter i andre medlemsorganisasjoner ellers                                           |19375|4   |
|100k_plus      |70.

## Persist findings

The JSON file stores the measured tables so report figures can be reproduced without relying on notebook display output.

In [6]:
def rows_to_records(dataframe):
    return [row.asDict(recursive=True) for row in dataframe.collect()]

findings = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_file": ANALYTICS_FILE,
    "rows": n_rows,
    "population_bands": rows_to_records(band_counts.drop("sort_order")),
    "legal_form_by_band": rows_to_records(legal_form_by_band),
    "financial_by_band": rows_to_records(financial_by_band),
    "distress_by_band": rows_to_records(distress_by_band),
    "industry_by_band": rows_to_records(industry_by_band),
}

with open(OUTPUT_FILE, "w") as output:
    json.dump(findings, output, indent=2, allow_nan=False, default=str)

print("Wrote geographic findings to", OUTPUT_FILE)

Wrote geographic findings to /home/jovyan/data/geography_analysis.json
